# Лабораторная работа 1 — EDA (разведочный анализ данных)

**Датасет:** Car Price Prediction (задача **регрессии**).  
**Целевая переменная:** `Selling_Price`.

Ноутбук соответствует разделу методички ЛР1 про разведочный анализ (п.11–12 задания).  
Структура по требованиям методички:
1. Загрузка данных и знакомство с ними
2. Очистка данных
3. Анализ признаков для модели
4. Сохранение финального датасета
5. Выводы


## Пункт задания / методички

**11.** Загрузить датасет в `./data`, создать в `./eda` интерактивный блокнот `eda.ipynb`.

**12.** Выполнить разведочный анализ. Блокнот структурировать по основным действиям:
- Загрузка данных и знакомство с ними
- Очистка данных
- Анализ признаков для модели
- Сохранение финального датасета
- Выводы

В загрузке и знакомстве показать основные статистики, сделать выводы о признаках, выделить столбцы с числовыми и категориальными признаками, выделить столбец с целевой переменной. Проверить типы данных каждого столбца и при необходимости откорректировать их.


### Реализация: импорты и пути к данным


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import plotly.figure_factory as ff
import seaborn as sns


def find_project_root() -> Path:
    """Ищем корень проекта по наличию data/dataset.csv (cwd может быть eda/ или корень)."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "dataset.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Не найден data/dataset.csv. Положите датасет в <project>/data/dataset.csv "
        f"(текущая директория: {here})"
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
EDA_DIR = PROJECT_ROOT / "eda"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)


## 1. Загрузка данных и знакомство с ними

### Пункт методички
Показать основные статистики, сделать выводы о признаках, выделить числовые и категориальные столбцы, выделить целевую переменную. Проверить типы и при необходимости откорректировать.


In [ ]:
df = pd.read_csv(DATA_DIR / "dataset.csv")
df.head(10)


In [ ]:
df.info(show_counts=True)
print("\nShape:", df.shape)
print("Duplicates:", df.duplicated().sum())
print("Missing values:\n", df.isna().sum())


In [ ]:
df.describe()


**Выводы по знакомству с данными:**
- Датасет содержит ≥300 объектов и 9 признаков (числовые и категориальные) — условие ЛР1 выполнено.
- **Целевая переменная:** `Selling_Price` (регрессия).
- **Числовые:** `Year`, `Present_Price`, `Driven_kms`, `Selling_Price`.
- **Категориальные:** `Car_Name`, `Fuel_Type`, `Selling_type`, `Transmission`, `Owner`.
- Пропусков нет; возможны дубликаты — удалим на этапе очистки.


## 2. Очистка данных

### Пункт методички
В очистке данных следует удалить невалидные данные, удалить или восстановить пропущенные значения.


In [ ]:
# Удаляем полные дубликаты
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed duplicates: {before - len(df)}")

# Корректируем типы
df["Fuel_Type"] = df["Fuel_Type"].astype("category")
df["Selling_type"] = df["Selling_type"].astype("category")
df["Transmission"] = df["Transmission"].astype("category")
df["Owner"] = df["Owner"].astype("category")
df["Car_Name"] = df["Car_Name"].astype("category")
df["Year"] = df["Year"].astype("int32")
df["Driven_kms"] = df["Driven_kms"].astype("int32")

# Фильтр невалидных значений
df = df[(df["Selling_Price"] > 0) & (df["Present_Price"] > 0) & (df["Driven_kms"] >= 0)]
df = df.reset_index(drop=True)

df.info(show_counts=True)


In [ ]:
cat_features = df.select_dtypes(include=["category"]).columns.to_list()
num_features = df.select_dtypes(include=["number"]).columns.to_list()
target = "Selling_Price"

print("Categorical:", cat_features)
print("Numeric:", num_features)
print("Target:", target)

for cat in cat_features:
    print(f"{cat} - nunique = {df[cat].nunique()}")


## 3. Анализ признаков для модели

### Пункт методички
Раздел должен содержать набор графиков, позволяющих сделать полезные выводы о данных и о зависимости целевой переменной от признаков.

Необходимо построить **не менее 5 графиков**, из которых **минимум один интерактивный**.

Каждый график сохранить в директорию `eda`. После каждого графика — вывод о выявленных зависимостях. На этом этапе могут быть сформированы новые признаки.


In [ ]:
# Новый признак: пробег в категориях low / mid / hi (по квантилям)
q33, q66 = df["Driven_kms"].quantile([0.33, 0.66])
df["Driven_run"] = df["Driven_kms"].apply(
    lambda x: "low" if x <= q33 else ("hi" if x > q66 else "mid")
).astype("category")
df[["Driven_kms", "Driven_run"]].head()


In [ ]:
# График 1
plt.figure(figsize=(8, 5))
sns.lineplot(data=df, x="Year", y="Selling_Price")
plt.title("Selling_Price vs Year")
plt.tight_layout()
plt.savefig(EDA_DIR / "graph1_price_by_year.png", dpi=120)
plt.show()
print("Вывод: более новые автомобили в среднем продаются дороже.")


In [ ]:
# График 2
g = sns.catplot(data=df, x="Fuel_Type", y="Selling_Price", kind="box")
g.fig.suptitle("Selling_Price by Fuel_Type", y=1.02)
g.savefig(EDA_DIR / "graph2_price_by_fuel.png", dpi=120)
plt.show()
print("Вывод: Diesel-автомобили в среднем имеют более высокую цену продажи, чем Petrol.")


In [ ]:
# График 3
df_car_price = (
    df.groupby("Car_Name", as_index=False, observed=True)
    .agg(mean_price=("Selling_Price", "mean"))
    .sort_values("mean_price", ascending=False)
    .head(25)
)
plt.figure(figsize=(10, 8))
sns.barplot(data=df_car_price, x="mean_price", y="Car_Name")
plt.title("Top-25 Car_Name by mean Selling_Price")
plt.tight_layout()
plt.savefig(EDA_DIR / "graph3_mean_price_by_car.png", dpi=120)
plt.show()
print("Вывод: модель автомобиля сильно влияет на цену; Car_Name — полезный категориальный признак.")


In [ ]:
# График 4
g = sns.displot(data=df, x="Selling_type", y="Selling_Price")
g.fig.suptitle("Selling_Price vs Selling_type", y=1.02)
g.savefig(EDA_DIR / "graph4_price_by_selling_type.png", dpi=120)
plt.show()
print("Вывод: у Dealer цены обычно выше, чем у Individual.")


In [ ]:
# График 5
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="Present_Price", y="Selling_Price", hue="Transmission")
plt.title("Selling_Price vs Present_Price")
plt.tight_layout()
plt.savefig(EDA_DIR / "graph5_price_vs_present.png", dpi=120)
plt.show()
print("Вывод: Present_Price имеет сильную положительную связь с Selling_Price.")


In [ ]:
# График 6 (интерактивный) + статичная копия для репозитория
corrs = df[["Selling_Price", "Year", "Driven_kms", "Present_Price"]].corr()
figure = ff.create_annotated_heatmap(
    z=corrs.values,
    x=list(corrs.columns),
    y=list(corrs.index),
    annotation_text=corrs.round(2).values,
    showscale=True,
)
figure.update_layout(title="Interactive correlation heatmap")
figure.write_html(EDA_DIR / "graph6_corr_interactive.html")

plt.figure(figsize=(6, 5))
sns.heatmap(corrs, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation heatmap")
plt.tight_layout()
plt.savefig(EDA_DIR / "graph6_corr.png", dpi=120)
plt.show()
figure.show()
print("Вывод: сильнейшая корреляция у Present_Price с Selling_Price; Driven_kms связан слабо отрицательно.")


## 4. Сохранение финального датасета

### Пункт методички
Финальный датасет после очистки сохранить. Рекомендуется формат **pickle**, чтобы сохранились типы столбцов.  
Файлы с данными в репозиторий **НЕ коммитятся**.


In [ ]:
clean_path = DATA_DIR / "clean_data.pkl"
df.to_pickle(clean_path)
print("Saved:", clean_path)

df_check = pd.read_pickle(clean_path)
df_check.info()


## 5. Выводы

### Пункт методички
В выводах указать:
- все действия этапа очистки;
- создавались ли новые признаки;
- какие закономерности по графикам полезны для решения задачи.

### Очистка данных
- удалены полные дубликаты;
- типы приведены к `category` / `int32`;
- отфильтрованы невалидные значения цены и пробега;
- пропусков в исходных данных не было.

### Новые признаки
- создан категориальный признак `Driven_run` (`low` / `mid` / `hi`) по квантилям `Driven_kms`.

### Закономерности для модели
- `Present_Price` — главный предиктор цены продажи;
- год выпуска положительно связан с ценой;
- Diesel и Dealer связаны с более высокой ценой;
- `Car_Name` и `Transmission` также влияют на цену;
- пробег влияет слабее, но полезен как категория `Driven_run`.

Выводы также перенесены в корневой `README.md` (п.13 задания ЛР1).
